### 小提琴+箱线图：展示3个人群的，表达量和甲基化差异，有充足统计学检验

In [ ]:
# ============================================================
# sig_gene_violin_final.py
# Panel A: Expression (4 genes, 1 row)
# Panel B: Methylation (7 genes, 2 rows)
# BH per comparison (over 142 genes), FDR<0.10, two-sided MWU
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ── Significant gene lists ───────────────────────────────────
expr_sig  = ['CD40LG', 'CYLD', 'IRF7', 'TNFRSF13C']
methy_sig = ['CD40LG', 'CD86', 'CLEC4D', 'FBXW11', 'IL1RAP', 'ITCH', 'LATS2']

palette    = {'No T2D': '#F1C40F', 'Pre-T2D': '#2ECC71', 'T2D': '#E74C3C'}
order      = ['No T2D', 'Pre-T2D', 'T2D']
stat_pairs = [('No T2D', 'Pre-T2D'), ('No T2D', 'T2D')]
x_pos_map  = {'No T2D': 0, 'Pre-T2D': 1, 'T2D': 2}

# ── Group labels ─────────────────────────────────────────────
train  = pd.read_csv("data/filtered_data/train_label_1.csv")
test   = pd.read_csv("data/filtered_data/test_label_1.csv")
labels = pd.concat([train, test], ignore_index=True)
labels.columns = ['subject_nodeidx', 't2ds', 'pret2ds', 'no_t2ds']
group_map = {
    row['subject_nodeidx']: (
        'T2D'     if row['t2ds']    == 1 else
        'Pre-T2D' if row['pret2ds'] == 1 else
        'No T2D'
    )
    for _, row in labels.iterrows()
}

# ── Load all 142 genes (required for valid BH correction) ───
all_genes = pd.read_csv(
    "analysis/gigtransformer-rownorm/inner_join_norm_refilter_node_weight_df_stable.csv"
)['gene_node_name'].tolist()

tran_all = pd.read_csv("data/filtered_data/merged_tran_v1_nodeidx_df.csv")
tran_genes_all = [g for g in all_genes if g in tran_all.columns]
tran_all = tran_all[['subject_nodeidx'] + tran_genes_all]

methy_files = [
    "data/filtered_data/merged_core_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_proximal_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_distal_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_upstream_nodeidx_df.csv",
    "data/filtered_data/merged_downstream_nodeidx_df.csv",
]
methy_dfs = [pd.read_csv(f) for f in methy_files]
methy_genes_all = [g for g in all_genes if g in methy_dfs[0].columns]
methy_all = pd.DataFrame({
    'subject_nodeidx': methy_dfs[0]['subject_nodeidx'].values,
    **{g: np.stack([df[g].values for df in methy_dfs], axis=1).mean(axis=1)
       for g in methy_genes_all}
})

# ── BH per comparison (over all 142 genes) ──────────────────
def mwu_bh_per_pair(df_wide, genes, group_map, pairs):
    result = {}
    for pair in pairs:
        ps = []
        valid = [g for g in genes if g in df_wide.columns]
        for gene in valid:
            df = df_wide[['subject_nodeidx', gene]].copy()
            df['Group'] = df['subject_nodeidx'].map(group_map)
            df = df.dropna(subset=['Group'])
            g1 = df[df['Group'] == pair[0]][gene].dropna().values
            g2 = df[df['Group'] == pair[1]][gene].dropna().values
            _, p = mannwhitneyu(g1, g2, alternative='two-sided')
            ps.append(p)
        _, qvals, _, _ = multipletests(ps, method='fdr_bh')
        for gene, q in zip(valid, qvals):
            result[(gene, pair)] = q
    return result

tran_q  = mwu_bh_per_pair(tran_all,  tran_genes_all,  group_map, stat_pairs)
methy_q = mwu_bh_per_pair(methy_all, methy_genes_all, group_map, stat_pairs)

# ── Helper functions ─────────────────────────────────────────
def q_to_star(q):
    if   q < 0.001: return '***'
    elif q < 0.01:  return '**'
    elif q < 0.05:  return '*'
    elif q < 0.10:  return '†'
    else:           return None

def cohens_d(g1, g2):
    ps = np.sqrt((np.std(g1, ddof=1)**2 + np.std(g2, ddof=1)**2) / 2)
    return (np.mean(g2) - np.mean(g1)) / ps if ps > 0 else 0.0

def draw_bracket(ax, x1, x2, y, star, eff_text, data_range):
    tick   = data_range * 0.012
    gap_up = data_range * 0.022
    gap_dn = data_range * 0.018
    xmid   = (x1 + x2) / 2
    ax.plot([x1, x2], [y, y],        color='#222', lw=1.2, clip_on=False)
    ax.plot([x1, x1], [y - tick, y], color='#222', lw=1.2, clip_on=False)
    ax.plot([x2, x2], [y - tick, y], color='#222', lw=1.2, clip_on=False)
    ax.text(xmid, y + gap_up, star,
            ha='center', va='bottom', fontsize=11, color='#111')
    ax.text(xmid, y - gap_dn, eff_text,
            ha='center', va='top', fontsize=7.5,
            style='italic', color='#333333')

def plot_panel(ax, df_wide, gene, group_map, qval_dict, data_type, ylabel=''):
    df = df_wide[['subject_nodeidx', gene]].copy()
    df['Group'] = df['subject_nodeidx'].map(group_map)
    df = df.dropna(subset=['Group'])

    sns.violinplot(data=df, x='Group', y=gene, order=order,
                   palette=palette, ax=ax,
                   inner=None, linewidth=1.2, alpha=0.55, cut=0)
    sns.boxplot(data=df, x='Group', y=gene, order=order,
                palette=palette, ax=ax,
                width=0.22, linewidth=1.2, fliersize=2.5,
                boxprops=dict(alpha=0.9),
                medianprops=dict(color='white', linewidth=2.0),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))

    y_max = df[gene].max()
    y_min = df[gene].min()
    data_range = max(y_max - y_min, 1e-6)
    bracket_y  = y_max + data_range * 0.10
    step       = data_range * 0.22

    for pair in stat_pairs:
        q    = qval_dict.get((gene, pair), 1.0)
        star = q_to_star(q)
        if star is None:
            continue
        g1 = df[df['Group'] == pair[0]][gene].values
        g2 = df[df['Group'] == pair[1]][gene].values
        if data_type == 'expression':
            eff_text = f"d={cohens_d(g1, g2):+.2f}"
        else:
            eff_text = f"ΔBeta={np.mean(g2)-np.mean(g1):+.3f}"
        draw_bracket(ax, x_pos_map[pair[0]], x_pos_map[pair[1]],
                     bracket_y, star, eff_text, data_range)
        bracket_y += step

    ax.set_ylim(y_min - data_range * 0.08, bracket_y + data_range * 0.08)
    ax.set_title(gene, fontsize=12, pad=4)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=4)
    ax.set_xticklabels([])
    ax.tick_params(axis='y', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_linewidth(1.1)
    ax.spines['bottom'].set_linewidth(1.1)

# ── Legend patches ───────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=palette['No T2D'],  label='No T2D'),
    mpatches.Patch(color=palette['Pre-T2D'], label='Pre-T2D'),
    mpatches.Patch(color=palette['T2D'],     label='T2D'),
]

# ════════════════════════════════════════════════════════════
# Panel A: Expression — 1 row × 4 cols
# ════════════════════════════════════════════════════════════
fig_a, axes_a = plt.subplots(
    nrows=1, ncols=4,
    figsize=(13, 4.5),
    gridspec_kw={'wspace': 0.42}
)

for col, gene in enumerate(expr_sig):
    ylabel = 'Expression\n(residual)' if col == 0 else ''
    plot_panel(axes_a[col], tran_all, gene, group_map, tran_q,
               'expression', ylabel=ylabel)
    axes_a[col].set_xticks([0, 1, 2])
    axes_a[col].set_xticklabels(['No\nT2D', 'Pre-\nT2D', 'T2D'],
                                fontsize=8.5)

fig_a.legend(handles=legend_patches, loc='lower center', ncol=3,
             fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig_a.suptitle(
    'RNA-seq Expression by Disease Group\n'
    '(† FDR<0.10  * FDR<0.05  ** FDR<0.01  *** FDR<0.001, BH-corrected MWU)',
    fontsize=11, y=1.02
)
fig_a.text(0.005, 0.5, 'A', fontsize=16,
           va='center', ha='left', transform=fig_a.transFigure)

fig_a.savefig("image_storage/sig_genes_expression_panelA.png",
              dpi=180, bbox_inches='tight')
plt.show()
print("Saved: Panel A (Expression)")

# ════════════════════════════════════════════════════════════
# Panel B: Methylation — 2 rows × 4 cols (7 panels, 1 hidden)
# ════════════════════════════════════════════════════════════
fig_b, axes_b = plt.subplots(
    nrows=2, ncols=4,
    figsize=(14, 9),
    gridspec_kw={'hspace': 0.55, 'wspace': 0.42}
)

for i, gene in enumerate(methy_sig):
    row, col = divmod(i, 4)
    ylabel = 'Methylation\n(\u03b2 avg)' if col == 0 else ''
    plot_panel(axes_b[row, col], methy_all, gene, group_map, methy_q,
               'methylation', ylabel=ylabel)
    if row == 1 or i >= 4:
        axes_b[row, col].set_xticks([0, 1, 2])
        axes_b[row, col].set_xticklabels(['No\nT2D', 'Pre-\nT2D', 'T2D'],
                                         fontsize=8.5)

# hide unused last panel
axes_b[1, 3].set_visible(False)

fig_b.legend(handles=legend_patches, loc='lower center', ncol=3,
             fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig_b.suptitle(
    'DNA Methylation (\u03b2 avg, 5-region) by Disease Group\n'
    '(† FDR<0.10  * FDR<0.05  ** FDR<0.01  *** FDR<0.001, BH-corrected MWU)',
    fontsize=11, y=1.02
)
fig_b.text(0.005, 0.97, 'B', fontsize=16,
           va='top', ha='left', transform=fig_b.transFigure)

fig_b.savefig("image_storage/sig_genes_methylation_panelB.png",
              dpi=180, bbox_inches='tight')
plt.show()
print("Saved: Panel B (Methylation)")



过时了

In [ ]:
# ============================================================
# core_gene_violin_v5.py
# V5: Method B — fresh two-sided Mann-Whitney for expression,
#     same approach as methylation, both BH-corrected.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ── 1. Top-12 genes ─────────────────────────────────────────
top12 = ['ERBIN','RNASEL','TNFSF13B','DHX33','TNFRSF13C','RNF125',
         'BEX3','LY96','CD40LG','CTNND1','TLR3','DCHS1']

palette    = {'No T2D': '#F1C40F', 'Pre-T2D': '#2ECC71', 'T2D': '#E74C3C'}
order      = ['No T2D', 'Pre-T2D', 'T2D']
stat_pairs = [('No T2D', 'Pre-T2D'), ('No T2D', 'T2D')]
x_pos_map  = {'No T2D': 0, 'Pre-T2D': 1, 'T2D': 2}

# ── 2. Group labels ─────────────────────────────────────────
train  = pd.read_csv("data/filtered_data/train_label_1.csv")
test   = pd.read_csv("data/filtered_data/test_label_1.csv")
labels = pd.concat([train, test], ignore_index=True)
labels.columns = ['subject_nodeidx', 't2ds', 'pret2ds', 'no_t2ds']
group_map = {
    row['subject_nodeidx']: (
        'T2D'     if row['t2ds']    == 1 else
        'Pre-T2D' if row['pret2ds'] == 1 else
        'No T2D'
    )
    for _, row in labels.iterrows()
}

# ── 3. Expression data ───────────────────────────────────────
tran = pd.read_csv("data/filtered_data/merged_tran_v1_nodeidx_df.csv")
# column name safety: keep subject_nodeidx + top12 cols
tran_cols = ['subject_nodeidx'] + [c for c in top12 if c in tran.columns]
tran = tran[tran_cols]

# ── 4. Methylation data (5-region mean) ──────────────────────
methy_files = [
    "data/filtered_data/merged_core_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_proximal_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_distal_promoter_nodeidx_df.csv",
    "data/filtered_data/merged_upstream_nodeidx_df.csv",
    "data/filtered_data/merged_downstream_nodeidx_df.csv",
]
methy_dfs = [pd.read_csv(f) for f in methy_files]
# average gene columns across 5 regions; use subject_nodeidx from first df
gene_cols = [c for c in top12 if c in methy_dfs[0].columns]
methy = methy_dfs[0][['subject_nodeidx']].copy()
for gene in gene_cols:
    vals = np.stack([df[gene].values for df in methy_dfs], axis=1)
    methy[gene] = vals.mean(axis=1)

# ── 5. Fresh two-sided Mann-Whitney + BH  (expression) ──────
def fresh_mwu_bh(df_wide, genes, group_map, pairs):
    """Returns dict {(gene, pair): q_value}"""
    records = []
    for gene in genes:
        df = df_wide[['subject_nodeidx', gene]].copy()
        df['Group'] = df['subject_nodeidx'].map(group_map)
        df = df.dropna(subset=['Group'])
        for pair in pairs:
            g1 = df[df['Group'] == pair[0]][gene].dropna().values
            g2 = df[df['Group'] == pair[1]][gene].dropna().values
            _, p = mannwhitneyu(g1, g2, alternative='two-sided')
            records.append((gene, pair, p))
    _, qvals, _, _ = multipletests([r[2] for r in records], method='fdr_bh')
    return {(r[0], r[1]): q for r, q in zip(records, qvals)}

tran_q  = fresh_mwu_bh(tran,  top12, group_map, stat_pairs)
methy_q = fresh_mwu_bh(methy, top12, group_map, stat_pairs)

# ── 6. Helper functions ──────────────────────────────────────
def q_to_star(q):
    if   q < 0.001: return '***'
    elif q < 0.01:  return '**'
    elif q < 0.05:  return '*'
    else:           return None

def cohens_d(g1, g2):
    ps = np.sqrt((np.std(g1, ddof=1)**2 + np.std(g2, ddof=1)**2) / 2)
    return (np.mean(g2) - np.mean(g1)) / ps if ps > 0 else 0.0

def draw_bracket(ax, x1, x2, y, star, eff_text, data_range):
    tick   = data_range * 0.012
    gap_up = data_range * 0.022
    gap_dn = data_range * 0.018
    xmid   = (x1 + x2) / 2
    ax.plot([x1, x2], [y, y],         color='#222', lw=1.2, clip_on=False)
    ax.plot([x1, x1], [y - tick, y],  color='#222', lw=1.2, clip_on=False)
    ax.plot([x2, x2], [y - tick, y],  color='#222', lw=1.2, clip_on=False)
    ax.text(xmid, y + gap_up, star,
            ha='center', va='bottom', fontsize=11, color='#111')
    ax.text(xmid, y - gap_dn, eff_text,
            ha='center', va='top', fontsize=7.5,
            style='italic', color='#333333')

def plot_panel(ax, df_wide, gene, group_map, qval_dict, data_type, ylabel=''):
    df = df_wide[['subject_nodeidx', gene]].copy()
    df['Group'] = df['subject_nodeidx'].map(group_map)
    df = df.dropna(subset=['Group'])

    sns.violinplot(data=df, x='Group', y=gene, order=order,
                   palette=palette, ax=ax,
                   inner=None, linewidth=1.2, alpha=0.55, cut=0)
    sns.boxplot(data=df, x='Group', y=gene, order=order,
                palette=palette, ax=ax,
                width=0.22, linewidth=1.2, fliersize=2.5,
                boxprops=dict(alpha=0.9),
                medianprops=dict(color='white', linewidth=2.0),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2))

    y_max = df[gene].max()
    y_min = df[gene].min()
    data_range = max(y_max - y_min, 1e-6)
    bracket_y  = y_max + data_range * 0.10
    step       = data_range * 0.22

    for pair in stat_pairs:
        q    = qval_dict.get((gene, pair), 1.0)
        star = q_to_star(q)
        if star is None:
            continue
        g1 = df[df['Group'] == pair[0]][gene].values
        g2 = df[df['Group'] == pair[1]][gene].values
        if data_type == 'expression':
            eff_text = f"d={cohens_d(g1, g2):+.2f}"
        else:
            eff_text = f"ΔBeta={np.mean(g2)-np.mean(g1):+.3f}"
        draw_bracket(ax, x_pos_map[pair[0]], x_pos_map[pair[1]],
                     bracket_y, star, eff_text, data_range)
        bracket_y += step

    ax.set_ylim(y_min - data_range * 0.08, bracket_y + data_range * 0.08)
    ax.set_title(gene, fontsize=13, fontweight='bold', pad=4)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel, fontsize=12, labelpad=4)
    ax.set_xticklabels([])
    ax.tick_params(axis='y', labelsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_linewidth(1.1)
    ax.spines['bottom'].set_linewidth(1.1)

# ── 7. Figure layout: 6 cols × 4 rows ───────────────────────
#   Row 0: Expression, genes  0-5
#   Row 1: Expression, genes  6-11
#   Row 2: Methylation, genes 0-5
#   Row 3: Methylation, genes 6-11
fig, axes = plt.subplots(
    nrows=4, ncols=6,
    figsize=(18, 14),
    gridspec_kw={'hspace': 0.60, 'wspace': 0.42}
)

for i, gene in enumerate(top12):
    row_tran  = i // 6          # 0 or 1
    row_methy = i // 6 + 2      # 2 or 3
    col       = i % 6

    # Expression
    ylabel_tran = 'Expression\n(residual)' if col == 0 else ''
    plot_panel(axes[row_tran, col], tran,  gene, group_map, tran_q,
               'expression', ylabel=ylabel_tran)

    # Methylation
    ylabel_meth = 'Methylation\n(β avg)' if col == 0 else ''
    plot_panel(axes[row_methy, col], methy, gene, group_map, methy_q,
               'methylation', ylabel=ylabel_meth)

# ── 8. x-tick labels (bottom of each block: rows 1 and 3) ───
for col in range(6):
    for row in [1, 3]:
        ax = axes[row, col]
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['No\nT2D', 'Pre-\nT2D', 'T2D'],
                           fontsize=8.5, rotation=0)

# ── 9. Legend ────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=palette['No T2D'],  label='No T2D'),
    mpatches.Patch(color=palette['Pre-T2D'], label='Pre-T2D'),
    mpatches.Patch(color=palette['T2D'],     label='T2D'),
]
fig.legend(handles=legend_patches,
           loc='lower center', ncol=3,
           fontsize=11, frameon=False,
           bbox_to_anchor=(0.5, -0.02))

# ── 10. Block labels (left margin) ───────────────────────────
fig.text(0.005, 0.78, 'RNA-seq\nExpression', va='center',
         ha='left', fontsize=12, fontweight='bold', rotation=90)
fig.text(0.005, 0.30, 'DNA\nMethylation', va='center',
         ha='left', fontsize=12, fontweight='bold', rotation=90)

# ── 11. Overall title ────────────────────────────────────────
fig.suptitle(
    'Top-12 Core Target Genes: Expression & Methylation by Disease Group\n'
    '(* FDR<0.05  ** FDR<0.01  *** FDR<0.001,  BH-corrected Mann-Whitney)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig("image_storage/core_gene_violin_expression_methylation_v5.png",
            dpi=180, bbox_inches='tight')
plt.show()
print("Saved: image_storage/core_gene_violin_expression_methylation_v5.png")



### 绘制圈富集结果图

In [ ]:
# ============================================================
# 143 GIG 基因富集分析 — 气泡图（Nature Methods 风格，第五次修订）
# 修订：去除所有加粗字体 / 每个面板单独输出为图片
# ============================================================
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.cm as cm
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.ticker import LogLocator, LogFormatterMathtext
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family':        'Arial',
    'font.size':          13.5,   # 9 × 1.5
    'axes.titlesize':     15,     # 10 × 1.5
    'axes.labelsize':     13.5,   # 9 × 1.5
    'xtick.labelsize':    12.75,  # 8.5 × 1.5
    'ytick.labelsize':    12.75,  # 8.5 × 1.5
    'axes.linewidth':     0.8,
    'xtick.major.width':  0.9,
    'ytick.major.width':  0.8,
    'xtick.major.size':   3.5,
    'ytick.major.size':   3,
    'axes.facecolor':     'white',
    'figure.facecolor':   'white',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'pdf.fonttype':       42,
    'svg.fonttype':       'none',
    'text.color':         '#000000',
    'axes.labelcolor':    '#000000',
    'xtick.color':        '#000000',
    'ytick.color':        '#000000',
    'axes.titlecolor':    '#000000',
})

PANEL_LABELS = ['a', 'b', 'c', 'd']

def add_panel_label(ax, label):
    ax.text(-0.40, 1.02, label, transform=ax.transAxes,
            fontsize=19.5, fontweight='normal', va='bottom', ha='left',  # 13 × 1.5
            clip_on=False, color='#000000')

def wrap_label(term, width=28):
    return '\n'.join(textwrap.wrap(term, width))

CMAP_ENRICH = LinearSegmentedColormap.from_list(
    'sig_navy', ['#E0ECF4', '#9EBCDA', '#08306B'], N=256
)

VMIN_C, VMAX_C = 1.0, 15.0
norm_c = Normalize(vmin=VMIN_C, vmax=VMAX_C)

# ────────────────────────────────────────────────────────────
# 1. 数据读取
# ────────────────────────────────────────────────────────────
df_raw = pd.read_excel('D:/LLFS-GiG-old/143GIGgene_enrichment_analysis.xlsx')
df = df_raw[df_raw['FDR'] < 0.05].copy()
df['neg_log10_FDR'] = -np.log10(df['FDR'])
df['Term_label']    = df['Term'].apply(wrap_label)

TOP_N = {
    'KEGG_PATHWAY':      15,
    'GOTERM_BP_DIRECT':  15,
    'GOTERM_MF_DIRECT':  10,
    'GOTERM_CC_DIRECT':  10,
}
PANEL_TITLES = {
    'KEGG_PATHWAY':      'KEGG Pathway',
    'GOTERM_BP_DIRECT':  'GO Biological Process',
    'GOTERM_MF_DIRECT':  'GO Molecular Function',
    'GOTERM_CC_DIRECT':  'GO Cellular Component',
}

subsets = {}
for cat, topn in TOP_N.items():
    sub = (df[df['Category'] == cat]
           .sort_values('FDR').head(topn)
           .sort_values('neg_log10_FDR', ascending=True)
           .copy())
    subsets[cat] = sub

count_max = df['Count'].max()
def bubble_size(count):
    return (count / count_max) ** 0.6 * 380 + 25

ROW_HEIGHT_PER_TERM = 0.44
FONTSIZE_LEG  = 11.25  # 7.5 × 1.5
legend_counts = [5, 20, 50, 100]

# ────────────────────────────────────────────────────────────
# 2. 每个面板单独绘图并保存
# ────────────────────────────────────────────────────────────
for idx, (cat, sub) in enumerate(subsets.items()):
    n_terms    = len(sub)
    fig_height = n_terms * ROW_HEIGHT_PER_TERM + 1.8

    fig, ax = plt.subplots(figsize=(8.5, fig_height))
    fig.subplots_adjust(left=0.38, right=0.88, top=0.93, bottom=0.08)

    x = sub['Fold Enrichment'].values
    y = np.arange(n_terms)

    ax.scatter(
        x, y,
        s=bubble_size(sub['Count'].values),
        c=sub['neg_log10_FDR'].values,
        cmap=CMAP_ENRICH, norm=norm_c,
        edgecolors='#444444', linewidths=0.5,
        alpha=0.92, zorder=3
    )

    ax.set_xscale('log')
    ax.xaxis.set_major_locator(LogLocator(base=10, numticks=6))
    ax.xaxis.set_major_formatter(LogFormatterMathtext(base=10))

    x_lo = max(x.min() * 0.40, 0.30)
    x_hi = x.max() * 2.20
    ax.set_xlim(x_lo, x_hi)

    if x_lo <= 1.0 <= x_hi:
        ax.axvline(x=1.0, color='#CCCCCC', linewidth=0.6,
                   linestyle=':', zorder=1, alpha=0.5)
    if x_lo <= 10.0 <= x_hi:
        ax.axvline(x=10.0, color='#AAAAAA', linewidth=0.8,
                   linestyle='--', zorder=1, alpha=0.6)

    ax.set_yticks(y)
    ax.set_yticklabels(sub['Term_label'].values,
                       fontsize=12.75, linespacing=1.2, color='#000000')  # 8.5 × 1.5
    ax.set_xlabel('Fold Enrichment', fontsize=13.5, color='#000000')      # 9 × 1.5
    ax.set_title(PANEL_TITLES[cat], fontsize=15, fontweight='normal',     # 10 × 1.5
                 pad=6, color='#000000')
    ax.grid(axis='x', linestyle=':', linewidth=0.4, alpha=0.35,
            color='#D3D3D3', zorder=0)
    ax.set_ylim(-0.8, n_terms - 0.2)

    ax.tick_params(axis='x', colors='#000000', width=0.9, labelsize=12.75)  # 8.5 × 1.5
    ax.tick_params(axis='y', colors='#000000', width=0.8, labelsize=12.75)  # 8.5 × 1.5

    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(0.8)
        ax.spines[spine].set_color('#333333')

    label = PANEL_LABELS[idx]
    fig.savefig(f'D:/LLFS-GiG-old/image_storage/143GIG_bubble_plot_{label}.png',
                dpi=1200, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)

# ────────────────────────────────────────────────────────────
# 3. 独立图例图
# ────────────────────────────────────────────────────────────
fig_leg, ax_leg = plt.subplots(figsize=(2.8, 3.2))
ax_leg.set_visible(False)

LEGEND_X    = 0.08
CBAR_W      = 0.12
CBAR_H      = 0.45
CBAR_BOTTOM = 0.42

cbar_ax = fig_leg.add_axes([LEGEND_X, CBAR_BOTTOM, CBAR_W, CBAR_H])
sm = cm.ScalarMappable(cmap=CMAP_ENRICH, norm=norm_c)
sm.set_array([])
cb = fig_leg.colorbar(sm, cax=cbar_ax)
cb.set_label(r'$-\log_{10}(\mathrm{FDR})$', fontsize=13.5, color='#000000')  # 9 × 1.5
cb.outline.set_linewidth(0.8)
cb.ax.tick_params(labelsize=12, width=0.8, color='#000000', labelcolor='#000000')  # 8 × 1.5
cb.ax.text(0.5, 1.03, f'≥{int(VMAX_C)}', transform=cb.ax.transAxes,
           fontsize=11.25, ha='center', va='bottom',                              # 7.5 × 1.5
           color='#000000', fontstyle='italic')

size_handles = [
    mlines.Line2D([], [], marker='o', linestyle='None',
                  markersize=np.sqrt(bubble_size(c)),
                  markerfacecolor='#aaaaaa', markeredgecolor='#444444',
                  markeredgewidth=0.5, alpha=0.9, label=f'n = {c}')
    for c in legend_counts
]
max_ms        = np.sqrt(bubble_size(max(legend_counts)))
borderpad_dyn = max_ms / (2 * FONTSIZE_LEG) + 0.4

leg = fig_leg.legend(
    handles=size_handles, title='Gene Count',
    title_fontsize=12, fontsize=FONTSIZE_LEG,          # 8 × 1.5 = 12
    loc='upper left',
    bbox_to_anchor=(LEGEND_X + 0.22, CBAR_BOTTOM - 0.05),
    bbox_transform=fig_leg.transFigure,
    frameon=True, framealpha=0.95,
    edgecolor='#cccccc', borderpad=borderpad_dyn, labelspacing=2.5
)
leg.get_title().set_color('#000000')
for t in leg.get_texts():
    t.set_color('#000000')

fig_leg.savefig('D:/LLFS-GiG-old/image_storage/143GIG_bubble_plot_legend.png',
                dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig_leg)



### p值4象限图

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from adjustText import adjust_text

# ---------- 数据 ----------
df = pd.read_csv(r'D:\LLFS-GiG-old\analysis\gigtransformer-rownorm\gene_quadrant_analysis.csv')
df['x'] = -np.log10(df['t2ds_no_t2ds_pvalue'])
df['y'] = -np.log10(df['pret2ds_no_t2ds_pvalue'])
df['avg_weight'] = (df['t2ds_weight'] + df['pret2ds_weight']) / 2

color_map = {
    'Both_sig':        '#D55E00',
    'T2D_specific':    '#0072B2',
    'PreT2D_specific': '#E69F00',
    'Non_sig':         '#F0F0F0',
}
ALPHA   = 0.05
sig_val = -np.log10(ALPHA)

# ---------- 画布 ----------
fig, ax = plt.subplots(figsize=(8.5, 7.8))
fig.patch.set_facecolor('white')

# ---------- 气泡 ----------
w_min, w_max = df['avg_weight'].min(), df['avg_weight'].max()
for quad in ['Non_sig', 'PreT2D_specific', 'T2D_specific', 'Both_sig']:
    sub = df[df['quadrant'] == quad]
    if quad == 'Non_sig':
        ax.scatter(sub['x'], sub['y'],
                   s=8, c=color_map[quad],
                   alpha=0.10,
                   edgecolors='none',
                   zorder=2)
    else:
        size = 25 + (sub['avg_weight'] - w_min) / (w_max - w_min + 1e-9) * 750
        ax.scatter(sub['x'], sub['y'],
                   s=size, c=color_map[quad],
                   alpha=0.82,
                   edgecolors='#333333',
                   linewidths=0.3,
                   zorder=4)

# ---------- 阈值参考线 ----------
ax.axvline(sig_val, color='#AAAAAA', linestyle='--', linewidth=0.8, zorder=1)
ax.axhline(sig_val, color='#AAAAAA', linestyle='--', linewidth=0.8, zorder=1)

# ---------- 坐标轴 ----------
ax.set_xlabel(r'$-\log_{10}(P\mathrm{-value})$  T2D vs No-T2D', fontsize=11)
ax.set_ylabel(r'$-\log_{10}(P\mathrm{-value})$  Pre-T2D vs No-T2D', fontsize=11)
ax.set_title('GiG Network Genes: Disease-Stage Specificity',
             fontsize=12, pad=10, fontweight='normal')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.7)
ax.spines['bottom'].set_linewidth(0.7)
ax.grid(False)

ax.set_xlim(-0.08, df['x'].max() * 1.13)
ax.set_ylim(-0.08, df['y'].max() * 1.13)

xticks = sorted(set(list(ax.get_xticks()) + [0.0]))
ax.set_xticks(xticks)

# ---------- 基因标注（无连接线） ----------
label_df = df[df['quadrant'] != 'Non_sig'].nlargest(22, 'avg_weight')
texts = []
for _, row in label_df.iterrows():
    t = ax.text(
        row['x'], row['y'], row['gene_node_name'],
        fontsize=7.2,
        fontweight='bold' if row['avg_weight'] > 0.25 else 'normal',
        color='#1a1a1a',
        zorder=5,
    )
    texts.append(t)

adjust_text(
    texts, ax=ax,
    # arrowprops 已移除，不绘制任何连接线
    expand_points=(2.0, 2.0),
    expand_text=(1.8, 1.8),
    force_text=1.0,
    force_points=0.8,
)

# ---------- 分类图例 ----------
quad_order = ['Both_sig', 'T2D_specific', 'PreT2D_specific', 'Non_sig']
label_map  = {
    'Both_sig':        'Both significant (n=25)',
    'T2D_specific':    'T2D-specific (n=45)',
    'PreT2D_specific': 'Pre-T2D-specific (n=19)',
    'Non_sig':         'Non-significant (n=56)',
}
sig_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=color_map[q],
           markeredgecolor='#333333' if q != 'Non_sig' else 'none',
           markeredgewidth=0.3,
           markersize=8, label=label_map[q])
    for q in quad_order
]
color_leg = ax.legend(
    handles=sig_handles,
    title='Significance',
    title_fontsize=8,
    loc='lower left',
    fontsize=8,
    frameon=False,
    borderpad=0.8,
)
ax.add_artist(color_leg)

# ---------- 气泡大小图例 ----------
size_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#999999',
           markeredgecolor='#333333', markeredgewidth=0.3,
           markersize=ms, label=lbl)
    for lbl, ms in [('Low (0.05)', 6), ('Medium (0.35)', 10), ('High (1.0)', 15)]
]
ax.legend(
    handles=size_handles,
    title='Network weight',
    title_fontsize=8,
    loc='lower right',
    fontsize=8,
    frameon=False,
    labelspacing=1.2,
    borderpad=0.8,
)
ax.add_artist(color_leg)

plt.tight_layout()
out = r'D:\LLFS-GiG-old\image_storage\gene_pvalue_quadrant_bubble_v4.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print('saved:', out)



四象限图V5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from adjustText import adjust_text

# ---------- 数据 ----------
df = pd.read_csv(r'D:\LLFS-GiG-old\analysis\gigtransformer-rownorm\gene_quadrant_analysis.csv')
df['x'] = -np.log10(df['t2ds_no_t2ds_pvalue'])
df['y'] = -np.log10(df['pret2ds_no_t2ds_pvalue'])
df['avg_weight'] = (df['t2ds_weight'] + df['pret2ds_weight']) / 2

color_map = {
    'Both_sig':        '#D55E00',
    'T2D_specific':    '#0072B2',
    'PreT2D_specific': '#E69F00',
    'Non_sig':         '#F0F0F0',
}
ALPHA   = 0.05
sig_val = -np.log10(ALPHA)

# ---------- 画布（保持尺寸，压缩边距）----------
fig, ax = plt.subplots(figsize=(8.5, 7.8))
fig.patch.set_facecolor('white')
fig.subplots_adjust(left=0.12, right=0.97, top=0.94, bottom=0.10)

# ---------- 气泡 ----------
w_min, w_max = df['avg_weight'].min(), df['avg_weight'].max()
for quad in ['Non_sig', 'PreT2D_specific', 'T2D_specific', 'Both_sig']:
    sub = df[df['quadrant'] == quad]
    if quad == 'Non_sig':
        ax.scatter(sub['x'], sub['y'],
                   s=10, c=color_map[quad],
                   alpha=0.10, edgecolors='none', zorder=2)
    else:
        size = 40 + (sub['avg_weight'] - w_min) / (w_max - w_min + 1e-9) * 900
        ax.scatter(sub['x'], sub['y'],
                   s=size, c=color_map[quad],
                   alpha=0.82, edgecolors='#333333', linewidths=0.4, zorder=4)

# ---------- 阈值参考线 ----------
ax.axvline(sig_val, color='#AAAAAA', linestyle='--', linewidth=1.0, zorder=1)
ax.axhline(sig_val, color='#AAAAAA', linestyle='--', linewidth=1.0, zorder=1)

# ---------- 坐标轴 ----------
ax.set_xlabel(r'$-\log_{10}(P\mathrm{-value})$  T2D vs No-T2D', fontsize=16)
ax.set_ylabel(r'$-\log_{10}(P\mathrm{-value})$  Pre-T2D vs No-T2D', fontsize=16)
ax.set_title('GiG Network Genes: Disease-Stage Specificity',
             fontsize=17, pad=8, fontweight='normal')
ax.tick_params(axis='both', labelsize=14)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.9)
ax.spines['bottom'].set_linewidth(0.9)
ax.grid(False)

ax.set_xlim(-0.08, df['x'].max() * 1.10)
ax.set_ylim(-0.08, df['y'].max() * 1.10)

xticks = sorted(set(list(ax.get_xticks()) + [0.0]))
ax.set_xticks(xticks)

# ---------- 基因标注（无连接线） ----------
label_df = df[df['quadrant'] != 'Non_sig'].nlargest(22, 'avg_weight')
texts = []
for _, row in label_df.iterrows():
    t = ax.text(
        row['x'], row['y'], row['gene_node_name'],
        fontsize=10.5,
        fontweight='normal',
        color='#1a1a1a',
        zorder=5,
    )
    texts.append(t)

adjust_text(
    texts, ax=ax,
    expand_points=(2.0, 2.0),
    expand_text=(1.8, 1.8),
    force_text=1.0,
    force_points=0.8,
)

# ---------- 分类图例（去掉 Non_sig） ----------
quad_order = ['Both_sig', 'T2D_specific', 'PreT2D_specific']
label_map  = {
    'Both_sig':        'Both significant (n=25)',
    'T2D_specific':    'T2D-specific (n=45)',
    'PreT2D_specific': 'Pre-T2D-specific (n=19)',
}
sig_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=color_map[q],
           markeredgecolor='#333333', markeredgewidth=0.4,
           markersize=13, label=label_map[q])
    for q in quad_order
]
color_leg = ax.legend(
    handles=sig_handles,
    title='Significance',
    title_fontsize=12,
    loc='lower left',
    fontsize=12,
    frameon=False,
    borderpad=0.6,
)
ax.add_artist(color_leg)

# ---------- 气泡大小图例 ----------
size_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#999999',
           markeredgecolor='#333333', markeredgewidth=0.4,
           markersize=ms, label=lbl)
    for lbl, ms in [('Low (0.05)', 7), ('Medium (0.35)', 12), ('High (1.0)', 17)]
]
ax.legend(
    handles=size_handles,
    title='Network weight',
    title_fontsize=12,
    loc='lower right',
    fontsize=12,
    frameon=False,
    labelspacing=1.0,
    borderpad=0.6,
)
ax.add_artist(color_leg)

out = r'D:\LLFS-GiG-old\image_storage\gene_pvalue_quadrant_bubble_v5.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print('saved:', out)



P值显著性对比图-不同人群

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# ---------- 数据 ----------
df = pd.read_csv(r'D:\LLFS-GiG-old\analysis\gigtransformer-rownorm\gene_quadrant_analysis.csv')
df['x'] = -np.log10(df['t2ds_no_t2ds_pvalue'])
df['y'] = -np.log10(df['pret2ds_no_t2ds_pvalue'])
df['avg_weight'] = (df['t2ds_weight'] + df['pret2ds_weight']) / 2

# 每组颜色映射用的显著性值
def pick_pval(row):
    if row['quadrant'] == 'T2D_specific':    return row['x']
    if row['quadrant'] == 'PreT2D_specific': return row['y']
    return (row['x'] + row['y']) / 2          # Both_sig 取均值

sig_df = df[df['quadrant'] != 'Non_sig'].copy()
sig_df['pval_color'] = sig_df.apply(pick_pval, axis=1)

# ---------- 面板配置 ----------
TOP_N = 20   # 每面板最多展示基因数（T2D_specific共45个，截取 top 20）

group_cfg = {
    'T2D_specific':    {
        'label':       'T2D-specific',
        'colors':      ['#D6E8F7', '#0072B2'],
        'cbar_label':  r'$-\log_{10}(P)$  T2D vs No-T2D',
    },
    'PreT2D_specific': {
        'label':       'Pre-T2D-specific',
        'colors':      ['#FFF3C0', '#E69F00'],
        'cbar_label':  r'$-\log_{10}(P)$  Pre-T2D vs No-T2D',
    },
    'Both_sig':        {
        'label':       'Both Significant',
        'colors':      ['#FFE5D0', '#D55E00'],
        'cbar_label':  r'$-\log_{10}(P)$  (T2D & Pre-T2D avg)',
    },
}
keys = ['T2D_specific', 'PreT2D_specific', 'Both_sig']

# 每组实际展示行数
ns = {k: min(TOP_N, len(sig_df[sig_df['quadrant'] == k])) for k in keys}

# ---------- 动态画布高度 ----------
ROW_H = 0.34    # 每基因占用高度（英寸）
PAD_H = 1.10    # 每面板额外高度
heights = [ns[k] * ROW_H + PAD_H for k in keys]
fig_h   = sum(heights) + 0.6

fig = plt.figure(figsize=(10, fig_h), facecolor='white')

# 主图列 + colorbar 列 × 3行
gs = gridspec.GridSpec(
    3, 2,
    height_ratios=heights,
    width_ratios=[1, 0.035],
    hspace=0.60,
    wspace=0.06,
    left=0.22, right=0.94,
    top=1 - 0.4 / fig_h,
    bottom=0.4 / fig_h,
)

# ---------- 逐面板绘制 ----------
for i, gkey in enumerate(keys):
    cfg = group_cfg[gkey]
    ax  = fig.add_subplot(gs[i, 0])
    cax = fig.add_subplot(gs[i, 1])

    # 筛选 & 按中心度降序排列（barh 从下到上，ascending=True）
    sub = (sig_df[sig_df['quadrant'] == gkey]
           .nlargest(ns[gkey], 'avg_weight')
           .sort_values('avg_weight', ascending=True)
           .reset_index(drop=True))

    # 颜色映射：颜色深度 = 显著性强度
    vmin, vmax = sub['pval_color'].min(), sub['pval_color'].max()
    norm  = Normalize(vmin=vmin, vmax=vmax)
    cmap  = LinearSegmentedColormap.from_list(gkey, cfg['colors'])
    colors = [cmap(norm(v)) for v in sub['pval_color']]

    # 水平条形图：条长 = 中心度权重
    ax.barh(range(len(sub)), sub['avg_weight'],
            color=colors,
            edgecolor='#444444', linewidth=0.25,
            height=0.72)

    # 条尾数值标签
    x_max = sub['avg_weight'].max()
    for j, val in enumerate(sub['avg_weight']):
        ax.text(val + x_max * 0.012, j, f'{val:.3f}',
                va='center', ha='left', fontsize=6.5, color='#555555')

    # Y 轴基因名
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub['gene_node_name'], fontsize=8)

    # 面板标题（左对齐 + 色彩呼应）
    total_n = len(sig_df[sig_df['quadrant'] == gkey])
    sfx = f'  (top {ns[gkey]} / {total_n})' if ns[gkey] < total_n else f'  (n = {ns[gkey]})'
    ax.set_title(cfg['label'] + sfx,
                 fontsize=10, fontweight='bold', loc='left', pad=5,
                 color=cfg['colors'][1])

    # 轴样式
    ax.set_xlim(0, x_max * 1.20)
    ax.set_xlabel('Network Centrality Weight', fontsize=8.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', length=0)   # 隐藏 Y 轴刻度线

    # Colorbar（右侧独立列）
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax)
    tick_vals = np.linspace(vmin, vmax, 4)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=6.5)
    cbar.set_label(cfg['cbar_label'], fontsize=7.5, labelpad=4)

# ---------- 总标题 ----------
fig.suptitle('GiG Hub Genes: Disease-Stage Specificity',
             fontsize=12, fontweight='normal',
             x=0.58, y=1 - 0.12 / fig_h)

out = r'D:\LLFS-GiG-old\image_storage\gene_bar_facet_v1.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor='white')
plt.close()
print('saved:', out)


### 按照故事，上中下游分模块的信号网络图

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# T2D 发病机制三层因果网络图  V4
# 重构：Convex Hull 平滑气泡 · 因果主干线 · 交错有机布局 · 菱形靶点 · 极简节点
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from scipy.spatial import ConvexHull
from scipy.interpolate import splprep, splev

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ─── 1. 读取数据 ──────────────────────────────────────────────────────────────
BASE = r"D:\LLFS-GiG-old\analysis\gigtransformer-rownorm"
node_df = pd.read_csv(f"{BASE}/inner_join_norm_refilter_node_weight_df.csv", index_col=0)
edge_df = pd.read_csv(f"{BASE}/inner_join_norm_refilter_edge_weight_df.csv", index_col=0)

idx2name    = dict(zip(node_df['gene_node_idx'],  node_df['gene_node_name']))
name2idx    = {v: k for k, v in idx2name.items()}
name2weight = dict(zip(node_df['gene_node_name'], node_df['Weight1']))

# ─── 2. 三层基因 ──────────────────────────────────────────────────────────────
LAYERS = {
    'upper':  ['ERBIN', 'RNASEL', 'DHX33', 'NOD2', 'PSTPIP1'],
    'middle': ['TNFSF13B', 'TNFRSF13C', 'LY96', 'CD40LG', 'TLR3', 'IL1R1', 'IL1B'],
    'lower':  ['BID', 'CASP4', 'NAIP', 'CARD8', 'CASP5', 'CARD16', 'GSDMD'],
}
missing = [g for l in LAYERS.values() for g in l if g not in name2weight]
if missing:
    print(f"[Warning] 跳过: {missing}")
    for layer in LAYERS:
        LAYERS[layer] = [g for g in LAYERS[layer] if g in name2weight]

DRUG_TARGETS = {'TNFSF13B', 'IL1B', 'IL1R1', 'CASP4'}

# ─── 3. 视觉配置（新配色）────────────────────────────────────────────────────
LAYER_Y     = {'upper': 3.0, 'middle': 1.5, 'lower': 0.0}
LAYER_COLOR = {'upper': '#4DBBD5', 'middle': '#F39B7F', 'lower': '#DC0000'}
LAYER_LABEL = {
    'upper':  'Metabolic Sensing',
    'middle': 'Immune Amplification',
    'lower':  'Cell Death',
}

# ─── 4. 节点坐标：权重排序 + 双行交错（赋予 Y 维度散布，避免纯水平线排列）─
X_SPACING = {'upper': 2.2, 'middle': 1.85, 'lower': 1.85}
ROW_SEP   = 0.30

def staggered_positions(genes, y_center, name2weight, x_spacing, row_sep):
    ordered = sorted(genes, key=lambda g: name2weight.get(g, 0), reverse=True)
    n = len(ordered)
    positions = {}
    for i, g in enumerate(ordered):
        x     = (i - (n - 1) / 2) * x_spacing
        y_off = row_sep / 2 if i % 2 == 0 else -row_sep / 2
        positions[g] = (x, y_center + y_off)
    return positions

pos = {}
for layer, genes in LAYERS.items():
    pos.update(staggered_positions(genes, LAYER_Y[layer], name2weight,
                                    X_SPACING[layer], ROW_SEP))
gene2layer = {g: l for l, gs in LAYERS.items() for g in gs}

# ─── 5. 同层 GiG 实线边 ──────────────────────────────────────────────────────
sel_idx2name = {name2idx[g]: g for g in pos if g in name2idx}
solid_edges  = []
for _, row in edge_df.iterrows():
    a, b = int(row['Actual_From']), int(row['Actual_To'])
    if a in sel_idx2name and b in sel_idx2name:
        ga, gb = sel_idx2name[a], sel_idx2name[b]
        if gene2layer.get(ga) == gene2layer.get(gb):
            w = (row['Weight1'] + row['Weight2']) / 2
            solid_edges.append((ga, gb, w, gene2layer[ga]))

seen, dedup = set(), []
for e in solid_edges:
    key = tuple(sorted([e[0], e[1]]))
    if key not in seen:
        seen.add(key)
        dedup.append(e)
solid_edges = dedup
print(f"[Info] 同层实线边 {len(solid_edges)} 条")

# ─── 6. 因果主干 ──────────────────────────────────────────────────────────────
CAUSAL_PATH       = ['NOD2', 'TNFSF13B', 'IL1B', 'CASP4']
CAUSAL_PAIRS_SORT = {tuple(sorted([CAUSAL_PATH[i], CAUSAL_PATH[i+1]]))
                     for i in range(len(CAUSAL_PATH) - 1)}
PATH_REF = {
    ('NOD2',     'TNFSF13B'): 'KEGG',
    ('TNFSF13B', 'IL1B'):     'Exp.Mol.Med.',
    ('IL1B',     'CASP4'):    'PMC 2024',
}

# ─── 7. 平滑凸包气泡 ─────────────────────────────────────────────────────────
def smooth_hull_blob(gene_list, pos_dict, pad=0.55, n_out=300):
    pts = np.array([pos_dict[g] for g in gene_list if g in pos_dict])
    if len(pts) < 3:
        cx, cy = pts.mean(axis=0)
        ang = np.linspace(0, 2 * np.pi, 50)
        return np.c_[cx + pad * np.cos(ang), cy + pad * 0.6 * np.sin(ang)]

    hull  = ConvexHull(pts)
    verts = pts[hull.vertices]
    cx, cy = verts.mean(axis=0)

    expanded = []
    for v in verts:
        d  = v - np.array([cx, cy])
        nm = np.linalg.norm(d)
        expanded.append(v + (d / nm if nm > 1e-9 else np.array([1.0, 0.0])) * pad)

    # 若顶点不足 4 个，插入中点保证三次样条可用
    while len(expanded) < 4:
        extra = []
        for i in range(len(expanded)):
            extra.append(expanded[i])
            extra.append((np.array(expanded[i]) +
                          np.array(expanded[(i + 1) % len(expanded)])) / 2)
        expanded = extra

    exp_arr = np.array(expanded)
    try:
        tck, _ = splprep([exp_arr[:, 0], exp_arr[:, 1]],
                          s=0.0, per=True, k=3)
        xn, yn = splev(np.linspace(0, 1, n_out), tck)
        return np.c_[xn, yn]
    except Exception:
        return np.vstack([exp_arr, exp_arr[0]])

# ─── 8. 绘图主体 ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 11))
ax.set_xlim(-9.5, 10.5)
ax.set_ylim(-1.2, 4.5)
ax.axis('off')
fig.patch.set_facecolor('white')

# ── 8a. 平滑 Convex Hull 气泡（填充极淡 + 半透明边框）
for layer, genes in LAYERS.items():
    blob  = smooth_hull_blob(genes, pos, pad=0.58)
    color = LAYER_COLOR[layer]
    ax.fill(blob[:, 0], blob[:, 1], color=color, alpha=0.10, zorder=0)
    ax.plot(blob[:, 0], blob[:, 1], color=color, alpha=0.50, lw=1.5, zorder=1)

# ── 8b. 模块副标题（气泡左上角，12pt italic）
for layer, genes in LAYERS.items():
    layer_pts = np.array([pos[g] for g in genes if g in pos])
    lx = layer_pts[:, 0].min() - 0.15
    ly = layer_pts[:, 1].max() + 0.65
    ax.text(lx, ly, LAYER_LABEL[layer],
            fontsize=12, style='italic',
            color=LAYER_COLOR[layer], fontweight='bold',
            va='bottom', ha='left', zorder=10)

# ── 8c. 层级流向箭头（右侧纵轴，极淡）
ax_arrow_x = 8.9
for ly_from, ly_to, label in [
    ('upper',  'middle', 'NF-κB activation'),
    ('middle', 'lower',  'Pyroptosis trigger'),
]:
    yf = LAYER_Y[ly_from] - 0.62
    yt = LAYER_Y[ly_to]   + 0.62
    ax.annotate('', xy=(ax_arrow_x, yt), xytext=(ax_arrow_x, yf),
                arrowprops=dict(arrowstyle='->', color='#BBBBBB',
                                lw=1.2, mutation_scale=14), zorder=2)
    ax.text(ax_arrow_x + 0.16, (yf + yt) / 2, label,
            va='center', ha='left', fontsize=8.5,
            color='#AAAAAA', style='italic')

# ── 8d. 同层 GiG 实线（极细浅灰 0.5pt；跳过因果主干对）
for ga, gb, w, layer in solid_edges:
    if tuple(sorted([ga, gb])) in CAUSAL_PAIRS_SORT:
        continue
    x1, y1 = pos[ga]
    x2, y2 = pos[gb]
    ax.plot([x1, x2], [y1, y2], '-',
            color='#DDDDDD', alpha=0.85, lw=0.5,
            zorder=2, solid_capstyle='round')

# ── 8e. 因果主干线（2pt 深灰实线 + 明确箭头 + 文献标注）
for i in range(len(CAUSAL_PATH) - 1):
    ga, gb = CAUSAL_PATH[i], CAUSAL_PATH[i + 1]
    if ga not in pos or gb not in pos:
        print(f"[Skip causal] {ga}→{gb}")
        continue
    x1, y1 = pos[ga]
    x2, y2 = pos[gb]
    arr = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle='-|>',
        mutation_scale=18,
        color='#444444',
        linewidth=2.0,
        connectionstyle='arc3,rad=0.06',
        shrinkA=11, shrinkB=11,
        zorder=5,
    )
    ax.add_patch(arr)
    ref = PATH_REF.get((ga, gb), '')
    if ref:
        ax.text((x1 + x2) / 2 + 0.28, (y1 + y2) / 2,
                ref, fontsize=7.5, color='#777777',
                style='italic', va='center', ha='left', zorder=6)

# ── 8f. 节点（普通=圆形；药物靶点=菱形；无内部数字）
MAX_W = max(name2weight.get(g, 0.01) for g in pos)
for gene, (x, y) in pos.items():
    layer = gene2layer[gene]
    w     = name2weight.get(gene, 0.01)
    size  = (w / MAX_W) ** 0.5 * 520 + 90
    color = LAYER_COLOR[layer]
    is_dt = gene in DRUG_TARGETS

    ax.scatter(x, y,
               s     = size * (1.3 if is_dt else 1.0),
               c     = color,
               marker= 'D' if is_dt else 'o',
               edgecolors='white', linewidths=1.5,
               zorder=7, alpha=0.93)

    ax.text(x, y - 0.18, gene,
            ha='center', va='top', fontsize=9.5,
            fontweight='bold', color='#111111', zorder=8)

# ── 8g. 图例
legend_handles = [
    mpatches.Patch(fc=LAYER_COLOR['upper'],  alpha=0.35,
                   ec=LAYER_COLOR['upper'],  lw=1.2,
                   label='Metabolic Sensing module'),
    mpatches.Patch(fc=LAYER_COLOR['middle'], alpha=0.35,
                   ec=LAYER_COLOR['middle'], lw=1.2,
                   label='Immune Amplification module'),
    mpatches.Patch(fc=LAYER_COLOR['lower'],  alpha=0.35,
                   ec=LAYER_COLOR['lower'],  lw=1.2,
                   label='Cell Death Execution module'),
    Line2D([0], [0], color='#DDDDDD', lw=0.8,
           label='GiG within-layer edge (width ∝ weight)'),
    Line2D([0], [0], color='#444444', lw=2.0,
           label='Causal trunk: NOD2 → TNFSF13B → IL1B → CASP4'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#888888', markersize=10,
           label='Gene node  (size ∝ GiG weight)'),
    Line2D([0], [0], marker='D', color='w',
           markerfacecolor='#888888', markersize=10,
           label='Known drug target  (diamond)'),
]
ax.legend(handles=legend_handles,
          loc='lower right',
          fontsize=8.8, framealpha=0.95, edgecolor='#dddddd',
          fancybox=False, borderpad=0.9, handlelength=2.0)

# ── 8h. 标题
ax.set_title(
    'T2D Pathogenesis: Three-Layer Mechanistic Gene Network  (V4)\n'
    'GiG Model Weights  ×  KEGG / Literature Causal Evidence',
    fontsize=13.5, fontweight='bold', pad=14, color='#111111'
)

# ─── 9. 保存 ──────────────────────────────────────────────────────────────────
out = r"D:\LLFS-GiG-old\image_storage\mechanism_network_v4.png"
plt.tight_layout()
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"[Done] {out}")





In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# T2D 发病机制信号网络图  V4-R（修订版）
# 修订：拓扑中心化 · 跨簇生物边 · 标签防重叠 · 标准尺寸图例 · 顶刊配色 · 黑框图例
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

plt.rcParams['font.family']       = 'Arial'
plt.rcParams['axes.unicode_minus'] = False

# ─── 1. 读取数据 ───────────────────────────────────────────────────────────────
BASE = r"D:\LLFS-GiG-old\analysis\gigtransformer-rownorm"
node_df = pd.read_csv(f"{BASE}/inner_join_norm_refilter_node_weight_df.csv", index_col=0)
edge_df = pd.read_csv(f"{BASE}/inner_join_norm_refilter_edge_weight_df.csv", index_col=0)

idx2name    = dict(zip(node_df['gene_node_idx'], node_df['gene_node_name']))
name2weight = dict(zip(node_df['gene_node_name'], node_df['Weight1']))

# ─── 2. 簇定义（NEJM/Nature 顶刊四色）─────────────────────────────────────────
CLUSTERS = {
    'Danger Sensing':       {'genes': ['ERBIN', 'RNASEL', 'DHX33', 'NOD2', 'PSTPIP1'],
                             'color': '#4DBBD5'},
    'NF-κB Amplification':  {'genes': ['TNFSF13B', 'TNFRSF13C', 'LY96', 'CD40LG', 'TLR3'],
                             'color': '#E64B35'},
    'Cytokine Signaling':   {'genes': ['IL1R1', 'IL1B'],
                             'color': '#00A087'},
    'Cell Death Execution': {'genes': ['BID', 'CASP4', 'NAIP', 'CARD8',
                                       'CASP5', 'CARD16', 'GSDMD'],
                             'color': '#DC0000'},
}
DRUG_TARGETS = {'TNFSF13B', 'IL1B', 'IL1R1', 'CASP4'}

for cl in CLUSTERS:
    CLUSTERS[cl]['genes'] = [g for g in CLUSTERS[cl]['genes'] if g in name2weight]

all_genes    = [g for cl in CLUSTERS.values() for g in cl['genes']]
gene2cluster = {g: cl for cl, v in CLUSTERS.items() for g in v['genes']}
gene2color   = {g: CLUSTERS[gene2cluster[g]]['color'] for g in all_genes}
MAX_W        = max(name2weight[g] for g in all_genes)

# ─── 3. 构建图 ────────────────────────────────────────────────────────────────
G = nx.Graph()
G.add_nodes_from(all_genes)

name2idx = {v: k for k, v in idx2name.items()}
sel_idx  = {name2idx[g] for g in all_genes if g in name2idx}

for _, row in edge_df.iterrows():
    a, b = int(row['Actual_From']), int(row['Actual_To'])
    if a in sel_idx and b in sel_idx:
        ga, gb = idx2name.get(a), idx2name.get(b)
        if ga in all_genes and gb in all_genes and ga != gb:
            G.add_edge(ga, gb, weight=(row['Weight1'] + row['Weight2']) / 2)

# 簇内链式连接（保证簇内连通）
for cl_info in CLUSTERS.values():
    gg = cl_info['genes']
    for i in range(len(gg) - 1):
        if not G.has_edge(gg[i], gg[i + 1]):
            G.add_edge(gg[i], gg[i + 1], weight=0.15)

# 跨簇生物学关键边（让网络具备真实信号传导连通性）
BIO_CROSS_EDGES = [
    ('NOD2',     'TNFSF13B', 0.6),   # Danger → NF-κB
    ('TLR3',     'IL1R1',    0.5),   # NF-κB → Cytokine
    ('TNFSF13B', 'IL1B',     0.7),   # NF-κB → Cytokine
    ('IL1B',     'CASP4',    0.6),   # Cytokine → Cell Death
    ('IL1B',     'BID',      0.4),   # Cytokine → Cell Death
]
for ga, gb, w in BIO_CROSS_EDGES:
    if ga in all_genes and gb in all_genes and not G.has_edge(ga, gb):
        G.add_edge(ga, gb, weight=w)

# ─── 4. 力导向布局（收紧弹簧，向中心聚合）────────────────────────────────────
# k 缩小至 1.1，消除四角流放；足够迭代次数使布局稳定收敛
pos_raw = nx.spring_layout(G, k=1.1, iterations=250, seed=42, weight='weight')

# 标准化到 [-4, 4] 坐标范围
coords = np.array(list(pos_raw.values()))
cx, cy = coords[:, 0].mean(), coords[:, 1].mean()
scale  = max(coords.ptp(axis=0)) / 2 + 1e-6
pos    = {g: ((pos_raw[g][0] - cx) / scale * 4,
              (pos_raw[g][1] - cy) / scale * 4) for g in pos_raw}

# ─── 5. 节点大小 ──────────────────────────────────────────────────────────────
def node_size(gene, min_s=200, max_s=2400):
    return min_s + ((name2weight.get(gene, 0.01) / MAX_W) ** 0.55) * (max_s - min_s)

cluster_hubs = {cl: max(v['genes'], key=lambda g: name2weight.get(g, 0))
                for cl, v in CLUSTERS.items() if v['genes']}
hub_genes    = set(cluster_hubs.values()) | DRUG_TARGETS

# ─── 6. 绘图主体 ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10), facecolor='white')
# 主轴：留出顶部标题 + 右侧图例空间
ax = fig.add_axes([0.02, 0.04, 0.70, 0.88])
ax.set_aspect('equal')
ax.axis('off')

# ── 6a. 边（规范：#A9A9A9, lw=1.5, alpha=0.6；跨簇边略粗以表示信号流）
for u, v, data in G.edges(data=True):
    x1, y1 = pos[u]
    x2, y2 = pos[v]
    is_cross = gene2cluster[u] != gene2cluster[v]
    ax.plot([x1, x2], [y1, y2], '-',
            color='#808080' if is_cross else '#A9A9A9',
            lw=1.8 if is_cross else 1.2,
            alpha=0.60, zorder=1, solid_capstyle='round')

# ── 6b. 节点
node_sizes  = [node_size(g) for g in all_genes]
node_colors = [gene2color[g] for g in all_genes]
node_ec     = ['#111111' if g in DRUG_TARGETS else 'white' for g in all_genes]
node_ew     = [2.8 if g in DRUG_TARGETS else 1.0 for g in all_genes]
xs, ys      = zip(*[pos[g] for g in all_genes])

ax.scatter(xs, ys, s=node_sizes, c=node_colors,
           edgecolors=node_ec, linewidths=node_ew,
           zorder=3, alpha=0.92)

# ── 6c. 标签（Hub：粗体簇色，外置；其余：小灰字，白底衬托可读性）
cx_all = np.mean([pos[g][0] for g in all_genes])
cy_all = np.mean([pos[g][1] for g in all_genes])

for gene in all_genes:
    x, y     = pos[gene]
    color    = gene2color[gene]
    r_coord  = np.sqrt(node_size(gene)) / 420  # scatter 尺寸 → 坐标半径（经验值）
    is_hub   = gene in hub_genes

    if is_hub:
        dx, dy = x - cx_all, y - cy_all
        norm   = np.sqrt(dx**2 + dy**2) + 1e-6
        off    = r_coord + 0.38
        tx, ty = x + dx / norm * off, y + dy / norm * off
        ha = 'left' if dx >= 0 else 'right'
        va = 'bottom' if dy >= 0 else 'top'
        ax.text(tx, ty, gene, ha=ha, va=va,
                fontsize=10.5, fontweight='bold', color=color, zorder=6,
                bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.82))
    else:
        ax.text(x, y - r_coord - 0.10, gene,
                ha='center', va='top', fontsize=8, color='#444444', zorder=6,
                bbox=dict(boxstyle='round,pad=0.10', fc='white', ec='none', alpha=0.72))

# ── 6d. 标题（物理隔离于绘图区上方，不与节点重叠）
fig.text(0.02, 0.97,
         'T2D Pathogenesis Signal Network  (V4)',
         ha='left', va='top', fontsize=13, fontweight='bold', color='#111111')
fig.text(0.02, 0.935,
         'GiG Model Weights  ·  Cluster Coloring  ·  Hub Labeling',
         ha='left', va='top', fontsize=9.5, color='#555555')

# ─── 7. 右侧图例区（两个独立子轴，互不干扰）────────────────────────────────────

# ── 7a. 节点尺寸图例（三圆垂直排列，无三角框架，明确标注权重含义）
sz_ax = fig.add_axes([0.75, 0.58, 0.23, 0.33])
sz_ax.axis('off')
sz_ax.set_xlim(0, 1)
sz_ax.set_ylim(0, 1)
sz_ax.text(0.0, 1.00, 'Node size (GiG Weight)',
           fontsize=9, fontweight='bold', va='top', color='#222222')

SIZE_LEVELS = [1.00, 0.50, 0.15]
R_MAX       = 0.11
Y_CENTERS   = [0.74, 0.48, 0.22]

for lv, yc in zip(SIZE_LEVELS, Y_CENTERS):
    r    = R_MAX * (lv ** 0.55)
    wval = MAX_W * lv
    circ = mpatches.Circle((0.22, yc), r,
                            fc='#888888', ec='white', lw=0.8, zorder=3)
    sz_ax.add_patch(circ)
    sz_ax.text(0.22 + R_MAX + 0.08, yc, f'{wval:.2f}',
               ha='left', va='center', fontsize=8.5, color='#333333')

# ── 7b. 节点颜色/簇图例 + 黑框说明
cl_ax = fig.add_axes([0.75, 0.04, 0.23, 0.50])
cl_ax.axis('off')
cl_ax.set_xlim(0, 1)
cl_ax.set_ylim(0, 1)
cl_ax.text(0.0, 1.00, 'Node color (Cluster)',
           fontsize=9, fontweight='bold', va='top', color='#222222')

cl_items = list(CLUSTERS.items())
N        = len(cl_items) + 1   # +1 为黑框条目
Y_START  = 0.88
Y_STEP   = Y_START / N

for i, (cl_name, cl_info) in enumerate(cl_items):
    yc = Y_START - i * Y_STEP
    cl_ax.add_patch(mpatches.Circle((0.09, yc), 0.052,
                    fc=cl_info['color'], ec='white', lw=0.8))
    cl_ax.text(0.20, yc, cl_name,
               ha='left', va='center', fontsize=8, color='#222222')

# 黑框：药物靶点说明
yc_drug = Y_START - len(cl_items) * Y_STEP
cl_ax.add_patch(mpatches.Circle((0.09, yc_drug), 0.052,
                fc='#888888', ec='#111111', lw=2.2))
cl_ax.text(0.20, yc_drug, 'Drug target\n(black border)',
           ha='left', va='center', fontsize=8, color='#222222', linespacing=1.3)

# ─── 8. 保存 ──────────────────────────────────────────────────────────────────
out = r"D:\LLFS-GiG-old\image_storage\mechanism_network_v4.png"
plt.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"[Done] {out}")
